# QueryChat Response Style Experiment for AI Burnout Checkup Dashboard

This notebook evaluates potential user-facing controls for customizing the behavior of the QueryChat interface in the **AI Explorer** tab of the burnout_checkup dashboard.

The goal is to identify a control that meaningfully influences the large language model (LLM) responses while remaining intuitive for dashboard users. The selected control will be integrated into the application and documented in the project specification.

## Objective

The purpose of this experiment is to evaluate a user-facing control that affects how the LLM responds to user queries in the AI Explorer.

The milestone requirement specifies that the dashboard should include at least one control that modifies LLM behavior. Examples include:

- response verbosity controls
- response style selection
- scope restrictions on dataset usage

In this experiment, we evaluate **response style control** as the candidate feature. The goal is to determine whether different response styles lead to meaningfully different outputs and whether such differences improve the usability of the AI Explorer.

## Candidate Controls Considered

Three potential QueryChat controls were considered:

### 1. Verbosity Slider
Allows users to adjust the length of responses (concise to detailed).

**Pros**
- Easy to implement
- Familiar interaction pattern

**Cons**
- Mostly changes answer length rather than interpretation
- May not significantly alter the usefulness of responses

---

### 2. Response Style Dropdown
Allows users to choose how explanations are framed.

Options evaluated in this experiment:

- **Executive Summary** - concise insights oriented toward managerial decision-making
- **Analytical Explanation** - balanced explanation of patterns in the dataset
- **Technical Interpretation** - more precise, dataset-aware interpretation referencing variables

**Pros**
- Produces clearly different types of answers
- Supports different user audiences
- Easy to explain in the UI

---

### 3. Scope Restriction Toggle
Limits responses to only the currently filtered dataset.

**Pros**
- Strong data integrity control
- Aligns AI answers with dashboard filters

**Cons**
- Requires deeper integration between QueryChat and dashboard filtering
- Higher implementation complexity

---

Based on these considerations, we prioritized **Response Style** for experimentation because it provides meaningful behavioral differences while remaining straightforward to implement and evaluate.

## Experimental Setup

To evaluate response styles, we generated responses to a consistent set of representative user questions.

Each question was asked using three response styles:

- Executive Summary
- Analytical Explanation
- Technical Interpretation

The experiment uses the **employee burnout and productivity dataset** used by the dashboard. The dataset includes variables related to:

- job role
- experience
- AI tool usage
- task automation
- workload indicators
- productivity scores
- burnout risk scores
- work-life balance

The goal is to observe how different response styles influence the interpretation and communication of patterns in the dataset.

### Imports

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from chatlas import ChatAnthropic

### Load, merge, and view data

In [2]:
FEATURES_PATH = Path("../data/raw/ai_productivity_features.csv")
TARGETS_PATH = Path("../data/raw/ai_productivity_targets.csv")

features = pd.read_csv(FEATURES_PATH)
targets = pd.read_csv(TARGETS_PATH)

df = features.merge(targets, on="Employee_ID", how="left")
print("Table 1: Dataset Snapshot\n")
df.head()

Table 1: Dataset Snapshot



,Employee_ID,job_role,experience_years,ai_tool_usage_hours_per_week,tasks_automated_percent,manual_work_hours_per_week,learning_time_hours_per_week,deadline_pressure_level,meeting_hours_per_week,collaboration_hours_per_week,error_rate_percent,task_complexity_score,focus_hours_per_day,work_life_balance_score,burnout_risk_score,productivity_score,burnout_risk_level
0,3c6ca882-3fa3-446b-8208-c92f3f306f06,Writer,19,11.8,28.5,19.2,1.4,High,1.9,2.3,0.20,2,7.1,4.8,10.00,81.0,High
1,02f168cc-7747-4dbd-a868-ea2cfb41e22a,Designer,4,10.8,24.1,23.3,2.6,Low,8.0,9.8,1.82,3,3.4,5.5,6.78,59.2,Medium
2,d39ce8c9-6e2a-4f86-b888-e2b5f4a18cf7,Developer,6,25.9,69.4,10.0,1.4,Medium,6.8,8.9,5.52,5,4.6,3.8,9.66,62.4,High
3,14511660-d78a-453f-9449-f17cd239ec27,Manager,20,7.9,17.2,25.1,0.2,High,3.5,8.6,1.14,5,5.6,3.9,10.00,76.8,High
4,0597f0bb-ed5a-4e35-94ac-3f0f6a5c2bc2,Developer,15,8.6,20.6,20.1,1.4,Low,5.9,5.3,2.75,10,1.0,7.4,5.38,53.7,Medium


### Define the dataset description

In [3]:
data_description = """
    Employee-level workplace wellbeing and productivity dataset.
    
    Each row represents one employee.
    
    Columns:
    - Employee_ID: unique identifier for each employee.
    - job_role: employee job role/category.
    - experience_years: years of experience.
    - ai_tool_usage_hours_per_week: hours per week spent using AI tools.
    - tasks_automated_percent: percent of tasks automated with AI/tools.
    - manual_work_hours_per_week: hours per week spent on manual work.
    - learning_time_hours_per_week: hours per week spent learning new tools/workflows.
    - deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
    - meeting_hours_per_week: hours per week spent in meetings.
    - collaboration_hours_per_week: hours per week spent collaborating.
    - error_rate_percent: percentage error rate.
    - task_complexity_score: task complexity score.
    - focus_hours_per_day: average focus/deep work hours per day.
    - work_life_balance_score: numeric work-life balance score.
    - burnout_risk_score: numeric burnout risk score.
    - productivity_score: numeric productivity score.
    - burnout_risk_level: burnout category label.
    
    Use this dataset to analyze:
    - burnout risk by role
    - relationships between AI usage and burnout
    - productivity vs burnout
    - work-life balance patterns
    - workload-related drivers of burnout
    
    Important:
    - Stay grounded in the available dataset columns.
    - Do not make causal claims.
    - Frame findings as associations, patterns, or comparisons.
"""

### Create a compact dataset summary for the prompt

This helps the model stay grounded.

In [4]:
def build_dataset_snapshot(df: pd.DataFrame) -> str:
    role_counts = df["job_role"].value_counts().to_dict()
    burnout_counts = df["burnout_risk_level"].value_counts().to_dict()

    summary = {
        "n_rows": int(len(df)),
        "job_roles": role_counts,
        "burnout_levels": burnout_counts,
        "avg_ai_usage_hours": round(
            df["ai_tool_usage_hours_per_week"].mean(),
            2
        ),
        "avg_manual_hours": round(
            df["manual_work_hours_per_week"].mean(), 2),
        "avg_burnout_score": round(
            df["burnout_risk_score"].mean(), 
            2
        ),
        "avg_productivity_score": round(df["productivity_score"].mean(), 2),
        "avg_work_life_balance": round(
            df["work_life_balance_score"].mean(),
            2
        ),
    }

    lines = ["Dataset snapshot:"]
    for k, v in summary.items():
        lines.append(f"- {k}: {v}")
    return "\n".join(lines)

dataset_snapshot = build_dataset_snapshot(df)
print(dataset_snapshot)

Dataset snapshot:
- n_rows: 4500
- job_roles: {'Developer': 1115, 'Analyst': 892, 'Designer': 722, 'Marketer': 658, 'Manager': 652, 'Writer': 461}
- burnout_levels: {'High': 3303, 'Medium': 1087, 'Low': 110}
- avg_ai_usage_hours: 10.35
- avg_manual_hours: 22.37
- avg_burnout_score: 8.35
- avg_productivity_score: 64.95
- avg_work_life_balance: 4.72


### Define the style instructions

In [5]:
STYLE_INSTRUCTIONS = {
    "executive": 
    """
    Respond in an Executive Summary style.
    - Use plain, concise language.
    - Emphasize the main takeaway first.
    - Mention practical workplace implications.
    - Avoid technical jargon unless necessary.
    - Keep the answer brief and decision-oriented.
    """,
    "analytical": 
    """
    Respond in an Analytical Explanation style.
    - Explain the main patterns clearly.
    - Compare relevant groups or variables where useful.
    - Balance clarity and detail.
    - Keep the answer grounded in the dataset.
    - Do not overstate conclusions.
    """,
    "technical": """
    Respond in a Technical Interpretation style.
    - Refer explicitly to dataset variables where relevant.
    - Emphasize associations, not causality.
    - Mention limitations or ambiguity when appropriate.
    - Use more precise analytical language.
    - Keep the answer dataset-grounded and method-aware.
    """
}

### Define the evaluation questions

In [6]:
questions = [
    "Which job roles appear to have the highest burnout risk?",
    "How does AI tool usage relate to burnout risk in this dataset?",
    "Show employees with high burnout risk and low productivity, and summarize what they have in common.",
    "What workplace factors seem most associated with burnout in this dataset?"
]

### Initialize the model

In [7]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if not anthropic_key:
    raise ValueError("ANTHROPIC_API_KEY is not set in your environment.")

chat_model = ChatAnthropic(model="claude-sonnet-4-0")

### Build a reusable prompt function

In [8]:
def build_prompt(question: str, style_key: str, data_description: str, dataset_snapshot: str) -> str:
    return f"""
    You are an AI assistant for a workplace burnout analytics dashboard.
    
    {STYLE_INSTRUCTIONS[style_key]}
    
    Context about the dataset:
    {data_description}
    
    {dataset_snapshot}
    
    User question:
    {question}
    
    Instructions:
    - Answer using only the dataset context provided.
    - If the question suggests causality, reframe it as association.
    - If a precise subgroup is requested, describe how you would identify it from the dataset fields.
    - Be truthful about uncertainty.
    """

### Run the experiment

This generates one response per question per style.

In [9]:
def ask_model(question: str, style_key: str) -> str:
    prompt = build_prompt(
        question=question,
        style_key=style_key,
        data_description=data_description,
        dataset_snapshot=dataset_snapshot,
    )
    response = chat_model.chat(prompt)
    return str(response)

records = []

for question in questions:
    for style_key in ["executive", "analytical", "technical"]:
        print(f"Running: {style_key} | {question}")
        answer = ask_model(question, style_key)
        records.append(
            {
                "question": question,
                "style": style_key,
                "response": answer,
            }
        )

experiment_df = pd.DataFrame(records)
experiment_df

Running: executive | Which job roles appear to have the highest burnout risk?


<br>

## Executive Summary: Burnout Risk by Job Role

**Main Finding:** The dataset shows significant variation in burnout risk across job roles, but a detailed breakdown by role is needed to identify which specific positions face the highest risk.

**Key Observations:**
- Overall burnout levels are concerning: 73% of employees show high burnout risk, with only 2% at low risk
- Six job roles are represented: Developers (largest group at 1,115 employees), followed by Analysts, Designers, Marketers, Managers, and Writers
- Average burnout score across all roles is 8.35 out of 10, indicating elevated risk company-wide

**Analysis Needed:**
To determine which roles have highest burnout risk, we would examine:
- Average burnout scores by job role
- Distribution of high/medium/low burnout categories within each role
- Associated factors like manual work hours, deadline pressure, and work-life balance scores per role

**Workplace Implications:**
- Role-specific burnout patterns could inform targeted intervention strategies
- High-risk roles may need workload redistribution, additional support, or process improvements
- Current burnout levels suggest urgent need for organization-wide wellbeing initiatives

**Recommendation:** Conduct immediate role-by-role burnout analysis to prioritize intervention efforts and allocate resources to the most affected positions.

Running: analytical | Which job roles appear to have the highest burnout risk?


<br>

## Burnout Risk Patterns by Job Role

Based on the available dataset context, I cannot definitively rank which job roles have the highest burnout risk, as the summary statistics don't provide role-specific burnout breakdowns. However, I can outline the analytical approach and what patterns would be most informative.

### Current Dataset Overview
The dataset includes 4,500 employees across six job roles with varying representation:
- **Developers**: 1,115 employees (25% of workforce)
- **Analysts**: 892 employees (20%)
- **Designers**: 722 employees (16%)
- **Marketers**: 658 employees (15%)
- **Managers**: 652 employees (14%)
- **Writers**: 461 employees (10%)

### Burnout Risk Distribution
Company-wide burnout levels show concerning patterns:
- **High risk**: 73% of employees (3,303 individuals)
- **Medium risk**: 24% of employees (1,087 individuals)  
- **Low risk**: Only 2% of employees (110 individuals)

### Analytical Framework for Role Comparison
To identify high-risk roles, the analysis would examine:

**Primary Burnout Indicators by Role:**
- Average burnout risk scores across the six job categories
- Distribution of high/medium/low burnout classifications within each role
- Comparison of role-specific scores against the overall average of 8.35

**Supporting Work Pattern Analysis:**
- **Workload factors**: Manual work hours (avg 22.37), meeting hours, and collaboration time by role
- **Pressure indicators**: Deadline pressure levels and task complexity scores
- **Balance metrics**: Work-life balance scores (avg 4.72) and focus hours per day
- **Technology adoption**: AI tool usage patterns and automation percentages

### Expected Patterns to Investigate
Different roles likely show distinct burnout risk profiles based on their work characteristics. For example, roles with higher manual work requirements, frequent deadline pressures, or extensive meeting schedules might correlate with elevated burnout scores, while roles with greater task automation or flexibility might show different patterns.

A complete role-by-role analysis of these metrics would reveal which positions face the greatest burnout challenges and help identify the specific work factors most strongly associated with burnout risk in each role.

Running: technical | Which job roles appear to have the highest burnout risk?


...

Running: analytical | What workplace factors seem most associated with burnout in this dataset?


<br>

## Workplace Factors Associated with Burnout Risk

The dataset provides comprehensive workplace variables that can reveal patterns associated with burnout risk. While the available summary statistics don't show direct correlations, the data structure suggests several key areas where associations are likely to emerge.

### Workload and Time Management Factors

**Manual Work Intensity:**
The average of 22.37 manual work hours per week represents a substantial workload that likely varies significantly across employees. Those with higher `manual_work_hours_per_week` may show elevated `burnout_risk_scores`, particularly when combined with limited automation. The relationship between manual hours and `tasks_automated_percent` could reveal whether employees are stuck in inefficient workflows.

**Meeting and Collaboration Burden:**
Excessive time in `meeting_hours_per_week` and `collaboration_hours_per_week` may correlate with higher burnout risk, especially when it reduces `focus_hours_per_day` for deep work. This pattern could be particularly pronounced among Managers (652 employees) compared to individual contributor roles.

**Deadline Pressure Patterns:**
The categorical `deadline_pressure_level` variable (Low, Medium, High) likely shows strong associations with burnout outcomes. Employees experiencing "High" deadline pressure probably demonstrate elevated `burnout_risk_scores` and may cluster in the High burnout category (currently 73% of workforce).

### Technology Adoption and Learning Burden

**AI Tool Utilization Patterns:**
With an average of 10.35 hours per week in AI tool usage, there's likely significant variation across employees. Those with lower `ai_tool_usage_hours_per_week` may experience higher burnout if they're handling more manual processes, while those with higher usage might show different patterns depending on implementation effectiveness.

**Automation Efficiency:**
The `tasks_automated_percent` variable could reveal important associations - employees with low automation percentages despite available tools may experience higher frustration and burnout risk. The combination of high `learning_time_hours_per_week` with low task automation might indicate ineffective technology adoption.

### Work-Life Integration and Focus

**Balance Score Relationships:**
The current average `work_life_balance_score` of 4.72 suggests room for improvement across the organization. This metric likely shows strong negative associations with burnout risk - employees with lower balance scores probably cluster in higher burnout categories.

**Deep Work Capacity:**
`Focus_hours_per_day` represents a crucial factor for both productivity and wellbeing. Employees with fragmented attention and limited focus time may show elevated burnout risk, particularly in roles requiring concentrated work like Developers or Analysts.

### Performance and Error Patterns

**Quality and Stress Relationships:**
`Error_rate_percent` may show bidirectional associations with burnout - higher error rates could both contribute to stress and result from burnout-related fatigue. The relationship between errors, `task_complexity_score`, and burnout risk could reveal whether employees are overwhelmed by their responsibilities.

**Experience and Adaptation:**
`Experience_years` might moderate other associations - newer employees may show different burnout patterns related to learning curves, while experienced employees might face different stressors related to increased responsibilities or change fatigue.

### Role-Specific Association Patterns

The six job roles likely exhibit distinct burnout association patterns:

**Developers (1,115 employees):** May show associations between burnout and factors like focus time disruption, technical complexity, and AI tool adoption effectiveness.

**Managers (652 employees):** Likely demonstrate associations between meeting hours, collaboration demands, and burnout risk.

**Creative roles (Designers, Writers, Marketers):** May show unique patterns related to task complexity, deadline pressure, and creative focus requirements.

### Comparative Analysis Framework

To identify the strongest associations, analysis would compare:
- High vs. low burnout employees across all workplace variables
- Role-specific patterns to identify universal vs. job-specific factors
- Interaction effects between variables (e.g., high manual work + low automation)
- Threshold effects where certain combinations become particularly problematic

The current organizational context - with 73% high burnout risk and relatively low work-life balance scores - suggests that multiple factors are likely operating simultaneously rather than single dominant drivers. This points toward systemic workplace patterns rather than isolated issues.

Running: technical | What workplace factors seem most associated with burnout in this dataset?


<br>

## Technical Analysis: Workplace Factors Associated with Burnout Risk

### Variable Classification and Association Framework

The dataset contains 14 workplace variables that can be categorized into distinct domains for burnout association analysis with the dependent variables `burnout_risk_score` (continuous, μ = 8.35) and `burnout_risk_level` (categorical, 73.4% High).

### Primary Candidate Predictors

**Workload Variables:**
- `manual_work_hours_per_week` (μ = 22.37): Expected negative correlation with wellbeing outcomes
- `meeting_hours_per_week`, `collaboration_hours_per_week`: Time allocation efficiency metrics
- `focus_hours_per_day`: Attention fragmentation indicator, likely negatively associated with `burnout_risk_score`

**Technology Adoption Variables:**
- `ai_tool_usage_hours_per_week` (μ = 10.35): Potential protective factor if associated with reduced manual workload
- `tasks_automated_percent`: Process efficiency metric, hypothesized negative association with burnout
- `learning_time_hours_per_week`: Skill development overhead, potential positive association with burnout if excessive

**Stress Indicators:**
- `deadline_pressure_level` (ordinal categorical): High expected effect size for burnout associations
- `task_complexity_score`: Cognitive load metric
- `error_rate_percent`: Performance quality indicator with potential bidirectional associations

### Statistical Modeling Considerations

**Correlation Matrix Requirements:**
Pearson correlations between continuous predictors and `burnout_risk_score`, with Spearman rank correlations for ordinal variables (`deadline_pressure_level`). Point-biserial correlations for continuous predictors with dichotomized `burnout_risk_level` (High vs. Medium/Low).

**Multicollinearity Assessment:**
Expected high intercorrelations between workload variables (`manual_work_hours_per_week`, `meeting_hours_per_week`, `collaboration_hours_per_week`). Variance inflation factors (VIF) analysis required before multivariate regression modeling.

**Effect Size Quantification:**
Cohen's conventions for correlation interpretation:
- |r| ≥ 0.5: Large association
- |r| ≥ 0.3: Medium association  
- |r| ≥ 0.1: Small association

### Role-Stratified Analysis Framework

**Job Role Interaction Effects:**
The 6-level `job_role` variable (n = 461-1115 per category) may moderate workplace factor associations through:
- Role × `manual_work_hours_per_week` interactions (expected higher impact for individual contributors)
- Role × `meeting_hours_per_week` interactions (expected higher impact for Managers)
- Role × `ai_tool_usage_hours_per_week` interactions (potentially strongest for Developers/Analysts)

**Within-Role Variance Analysis:**
ANOVA decomposition to assess between-role vs. within-role variance in `burnout_risk_score` across workplace factors.

### Methodological Limitations

**Cross-Sectional Design Constraints:**
Employee-level observational data precludes temporal ordering. High correlations between workplace factors and `burnout_risk_score` may reflect:
- Workplace conditions → burnout risk
- Burnout status → altered work patterns
- Third-variable confounding (organizational culture, individual differences)

**Measurement Scale Ambiguity:**
Several variables lack specified ranges or validation information:
- `burnout_risk_score`: Scale boundaries undefined
- `work_life_balance_score` (μ = 4.72): Interpretation relative to scale maximum unclear
- `task_complexity_score`, `productivity_score`: Standardization unknown

**Self-Report Bias:**
Subjective variables (`focus_hours_per_day`, `ai_tool_usage_hours_per_week`) may exhibit systematic reporting biases, particularly among high-burnout employees.

### Expected Association Patterns

**High-Magnitude Associations (|r| ≥ 0.4):**
- `work_life_balance_score` ↔ `burnout_risk_score` (negative)
- `deadline_pressure_level` ↔ `burnout_risk_score` (positive)
- `manual_work_hours_per_week` ↔ `burnout_risk_score` (positive)

**Moderate Associations (0.2 ≤ |r| < 0.4):**
- `focus_hours_per_day` ↔ `burnout_risk_score` (negative)
- `error_rate_percent` ↔ `burnout_risk_score` (positive)
- `tasks_automated_percent` ↔ `burnout_risk_score` (negative)

### Advanced Analytical Approaches

**Principal Component Analysis:**
Dimensionality reduction of the 14 workplace variables to identify latent factors (e.g., "Workload Intensity," "Technology Adoption," "Work Quality") for parsimony in burnout prediction models.

**Machine Learning Feature Importance:**
Random forest or gradient boosting algorithms to rank variable importance for `burnout_risk_level` classification, accounting for non-linear associations and interaction effects not captured in correlation analysis.

**Mediation Analysis:**
Potential mediation pathways (e.g., `manual_work_hours_per_week` → `work_life_balance_score` → `burnout_risk_score`) using structural equation modeling approaches.

### Population Context Interpretation

The extreme `burnout_risk_level` distribution (97.6% Medium-High) suggests either:
- Severe organizational wellness crisis requiring immediate intervention
- Scale calibration issues resulting in restricted range
- Selection bias in survey participation

This distribution pattern limits discriminatory power for identifying workplace factors specific to high-burnout employees, as the majority of the sample falls into this category.

,question,style,response
0,Which job roles appear to have the highest bur...,executive,## Executive Summary: Burnout Risk by Job Role...
1,Which job roles appear to have the highest bur...,analytical,## Burnout Risk Patterns by Job Role\n\nBased ...
2,Which job roles appear to have the highest bur...,technical,## Technical Analysis: Role-Specific Burnout R...
3,How does AI tool usage relate to burnout risk ...,executive,## Executive Summary: AI Tool Usage and Burnou...
4,How does AI tool usage relate to burnout risk ...,analytical,## AI Tool Usage and Burnout Risk Associations...
5,How does AI tool usage relate to burnout risk ...,technical,## Technical Analysis: AI Tool Usage and Burno...
6,Show employees with high burnout risk and low ...,executive,"## Executive Summary: High Burnout, Low Produc..."
7,Show employees with high burnout risk and low ...,analytical,"## Analysis of High Burnout Risk, Low Producti..."
8,Show employees with high burnout risk and low ...,technical,"## Technical Analysis: High Burnout Risk, Low ..."
9,What workplace factors seem most associated wi...,executive,## Executive Summary: Key Workplace Factors As...


## Save the experiment results

In [10]:
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

experiment_path = output_dir / "querychat_response_style_experiments.csv"
experiment_df.to_csv(experiment_path, index=False)

experiment_path

WindowsPath('outputs/querychat_response_style_experiments.csv')

## Evaluation Criteria

Responses were evaluated qualitatively using the following criteria:

### Relevance
Does the response remain grounded in the available dataset fields?

### Clarity
Is the explanation understandable for dashboard users?

### Actionability
Does the answer provide useful insights that help interpret the dashboard?

### Audience Fit
Does the response style match a plausible user audience (e.g., managers vs analysts)?

### Faithfulness
Does the response avoid unsupported causal claims and remain consistent with the dataset context?

These criteria allow us to assess whether a response style meaningfully improves the interpretability of AI-generated explanations.

### Add a manual evaluation table

In [11]:
# Create evaluation table from experiment results
evaluation_template = experiment_df.copy()

# Add explicit scoring columns
# Scale: 1 = poor, 3 = acceptable, 5 = strong
evaluation_template["relevance_score"] = pd.NA
evaluation_template["clarity_score"] = pd.NA
evaluation_template["actionability_score"] = pd.NA
evaluation_template["audience_fit_score"] = pd.NA
evaluation_template["faithfulness_score"] = pd.NA

# Overall summary score for easier comparison later
evaluation_template["overall_score"] = pd.NA

# Short qualitative notes for what worked/did not work
evaluation_template["notes"] = ""

# Final recommendation flag per row
evaluation_template["recommended_for"] = ""

# Save blank evaluation template
evaluation_path = output_dir / "querychat_response_style_evaluation_template.csv"
evaluation_template.to_csv(evaluation_path, index=False)

print("Table 2: Response Style Evaluation Template\n")
evaluation_template.head()

Table 2: Response Style Evaluation Template



,question,style,response,relevance_score,clarity_score,actionability_score,audience_fit_score,faithfulness_score,overall_score,notes,recommended_for
0,Which job roles appear to have the highest bur...,executive,## Executive Summary: Burnout Risk by Job Role...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
1,Which job roles appear to have the highest bur...,analytical,## Burnout Risk Patterns by Job Role\n\nBased ...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
2,Which job roles appear to have the highest bur...,technical,## Technical Analysis: Role-Specific Burnout R...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
3,How does AI tool usage relate to burnout risk ...,executive,## Executive Summary: AI Tool Usage and Burnou...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
4,How does AI tool usage relate to burnout risk ...,analytical,## AI Tool Usage and Burnout Risk Associations...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,


In [12]:
evaluation_df = experiment_df.copy()

evaluation_df["relevance_score"] = pd.NA
evaluation_df["clarity_score"] = pd.NA
evaluation_df["actionability_score"] = pd.NA
evaluation_df["audience_fit_score"] = pd.NA
evaluation_df["faithfulness_score"] = pd.NA
evaluation_df["notes"] = ""

In [13]:
# Style-level scoring based on qualitative review of outputs
score_map = {
    ("Which job roles appear to have the highest burnout risk?", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 3,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Clear and accessible, but somewhat general."
    },
    ("Which job roles appear to have the highest burnout risk?", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 4,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Balanced and interpretable, with better grounding in role comparisons."
    },
    ("Which job roles appear to have the highest burnout risk?", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 3,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Most precise, but less readable for general users."
    },

    ("How does AI tool usage relate to burnout risk in this dataset?", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 3,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Readable summary, but may oversimplify the relationship."
    },
    ("How does AI tool usage relate to burnout risk in this dataset?", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 4,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Best balance of explanation and caution."
    },
    ("How does AI tool usage relate to burnout risk in this dataset?", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 3,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Strong variable awareness and caution, but less accessible."
    },

    ("Show employees with high burnout risk and low productivity, and summarize what they have in common.", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 4,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Useful summary, though not always specific enough."
    },
    ("Show employees with high burnout risk and low productivity, and summarize what they have in common.", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 5,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Strongest for subgroup interpretation and pattern explanation."
    },
    ("Show employees with high burnout risk and low productivity, and summarize what they have in common.", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 4,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Accurate and careful, but not as user-friendly."
    },

    ("What workplace factors seem most associated with burnout in this dataset?", "executive"): {
        "relevance_score": 4,
        "clarity_score": 5,
        "actionability_score": 3,
        "audience_fit_score": 4,
        "faithfulness_score": 3,
        "notes": "Concise but tends to stay high-level."
    },
    ("What workplace factors seem most associated with burnout in this dataset?", "analytical"): {
        "relevance_score": 5,
        "clarity_score": 4,
        "actionability_score": 4,
        "audience_fit_score": 5,
        "faithfulness_score": 4,
        "notes": "Clear explanation of likely associated factors without overclaiming."
    },
    ("What workplace factors seem most associated with burnout in this dataset?", "technical"): {
        "relevance_score": 5,
        "clarity_score": 3,
        "actionability_score": 3,
        "audience_fit_score": 3,
        "faithfulness_score": 5,
        "notes": "Most rigorous, but least approachable."
    },
}

In [14]:
evaluation_df = experiment_df.copy()

for col in [
    "relevance_score",
    "clarity_score",
    "actionability_score",
    "audience_fit_score",
    "faithfulness_score",
    "notes",
]:
    evaluation_df[col] = pd.NA

for i, row in evaluation_df.iterrows():
    key = (row["question"], row["style"])
    if key in score_map:
        for col, value in score_map[key].items():
            evaluation_df.loc[i, col] = value

score_cols = [
    "relevance_score",
    "clarity_score",
    "actionability_score",
    "audience_fit_score",
    "faithfulness_score",
]

evaluation_df["overall_score"] = (
    evaluation_df[score_cols].astype(float).mean(axis=1).round(2)
)

print("Table 3: Scored Response Style Evaluation Template\n")
evaluation_df

Table 3: Scored Response Style Evaluation Template



,question,style,response,relevance_score,clarity_score,actionability_score,audience_fit_score,faithfulness_score,notes,overall_score
0,Which job roles appear to have the highest bur...,executive,## Executive Summary: Burnout Risk by Job Role...,4,5,3,4,3,"Clear and accessible, but somewhat general.",3.8
1,Which job roles appear to have the highest bur...,analytical,## Burnout Risk Patterns by Job Role\n\nBased ...,5,4,4,5,4,"Balanced and interpretable, with better ground...",4.4
2,Which job roles appear to have the highest bur...,technical,## Technical Analysis: Role-Specific Burnout R...,5,3,3,3,5,"Most precise, but less readable for general us...",3.8
3,How does AI tool usage relate to burnout risk ...,executive,## Executive Summary: AI Tool Usage and Burnou...,4,5,3,4,3,"Readable summary, but may oversimplify the rel...",3.8
4,How does AI tool usage relate to burnout risk ...,analytical,## AI Tool Usage and Burnout Risk Associations...,5,4,4,5,4,Best balance of explanation and caution.,4.4
5,How does AI tool usage relate to burnout risk ...,technical,## Technical Analysis: AI Tool Usage and Burno...,5,3,3,3,5,"Strong variable awareness and caution, but les...",3.8
6,Show employees with high burnout risk and low ...,executive,"## Executive Summary: High Burnout, Low Produc...",4,5,4,4,3,"Useful summary, though not always specific eno...",4.0
7,Show employees with high burnout risk and low ...,analytical,"## Analysis of High Burnout Risk, Low Producti...",5,4,5,5,4,Strongest for subgroup interpretation and patt...,4.6
8,Show employees with high burnout risk and low ...,technical,"## Technical Analysis: High Burnout Risk, Low ...",5,3,4,3,5,"Accurate and careful, but not as user-friendly.",4.0
9,What workplace factors seem most associated wi...,executive,## Executive Summary: Key Workplace Factors As...,4,5,3,4,3,Concise but tends to stay high-level.,3.8


In [15]:
final_eval_path = output_dir / "querychat_response_style_evaluation_results.csv"
evaluation_df.to_csv(final_eval_path, index=False)

final_eval_path

WindowsPath('outputs/querychat_response_style_evaluation_results.csv')

## Experiment Results

For each evaluation question, responses were generated using the three response styles.

The outputs are compared to determine how the style affects:

- explanation depth
- dataset grounding
- clarity of insights
- usefulness for dashboard interpretation

The results tables below contain the generated responses and allow comparison across styles.

#### Generate Style Comparison Summary

In [16]:
score_cols = [
    "relevance_score",
    "clarity_score",
    "actionability_score",
    "audience_fit_score",
    "faithfulness_score",
]

# Force score columns to numeric
for col in score_cols + ["overall_score"]:
    evaluation_df[col] = pd.to_numeric(evaluation_df[col], errors="coerce")

style_summary_detailed = (
    evaluation_df.groupby("style")[score_cols + ["overall_score"]]
    .mean()
    .round(2)
    .sort_values("overall_score", ascending=False)
)

print("Table 4: Detailed Evaluation Scores by Response Style\n")
style_summary_detailed

Table 4: Detailed Evaluation Scores by Response Style



,relevance_score,clarity_score,actionability_score,audience_fit_score,faithfulness_score,overall_score
style,,,,,,
analytical,5.0,4.0,4.25,5.0,4.0,4.45
executive,4.0,5.0,3.25,4.0,3.0,3.85
technical,5.0,3.0,3.25,3.0,5.0,3.85


In [17]:
detailed_path = output_dir / "querychat_style_summary_detailed.csv"
style_summary_detailed.to_csv(detailed_path)

detailed_path

WindowsPath('outputs/querychat_style_summary_detailed.csv')

In [18]:
style_summary_compact = (
    evaluation_df.groupby("style")[["overall_score"]]
    .mean()
    .round(2)
    .sort_values("overall_score", ascending=False)
)

print("Table 5: Overall Response Style Comparison\n")
style_summary_compact

Table 5: Overall Response Style Comparison



,overall_score
style,
analytical,4.45
executive,3.85
technical,3.85


In [19]:
compact_path = output_dir / "querychat_style_summary_compact.csv"
style_summary_compact.to_csv(compact_path)

compact_path

WindowsPath('outputs/querychat_style_summary_compact.csv')

Tables 4 and 5 summarize the average evaluation scores for each response style across the experiment questions.

## Discussion

The responses generated using different styles showed clear differences in clarity, interpretability, and dataset grounding. These differences were evaluated using five criteria: relevance, clarity, actionability, audience fit, and faithfulness to the dataset.

**Executive Summary**  
- Achieved the highest clarity score (5.0), reflecting its concise and accessible explanations.  
- Useful for quick insights and decision-oriented summaries.  
- However, it received lower faithfulness and actionability scores (3.0 and 3.25), indicating that responses sometimes remained high-level and less grounded in specific dataset variables.

**Analytical Explanation**  
- Achieved the highest overall score (**4.45**) and performed consistently well across all evaluation criteria.  
- Particularly strong in relevance (5.0) and audience fit (5.0), demonstrating its ability to explain dataset patterns while remaining readable.  
- Provides the best balance between interpretability and analytical depth, making it the most suitable default response style for dashboard users.

**Technical Interpretation**  
- Achieved the highest faithfulness score (5.0), reflecting its precise references to variables and methodological caution.  
- However, it scored lower in clarity and audience fit (3.0 each), indicating that the responses may be harder for non-technical users to interpret.

Overall, the experiment shows that **Analytical Explanation provides the most balanced and useful responses**, while Executive Summary prioritizes accessibility and Technical Interpretation prioritizes precision.

## Final Decision

Based on the experimental evaluation, we selected a **Response Style dropdown** as the user-facing control for the QueryChat interface in the AI Explorer tab.

The experiment compared three candidate response styles: **Executive Summary**, **Analytical Explanation**, and **Technical Interpretation**. Each style was evaluated across five criteria: relevance, clarity, actionability, audience fit, and faithfulness to the dataset.

The results showed that **Analytical Explanation achieved the highest overall score (4.45)** and performed consistently well across all evaluation metrics. In particular, it demonstrated strong relevance to the dataset (5.0) and the best audience fit (5.0), while maintaining good clarity and interpretability.

By comparison:

- **Executive Summary** produced the clearest responses (clarity score of 5.0) and was well suited for quick insights, but often remained high-level and less grounded in dataset variables.
- **Technical Interpretation** provided the most precise and method-aware responses (faithfulness score of 5.0), but was less accessible for general dashboard users due to lower clarity and audience fit scores.

Given these results, **Analytical Explanation was selected as the default response style**, as it offers the best balance between interpretability, dataset grounding, and usefulness for typical dashboard users.

The final control implemented in the AI Explorer allows users to choose among three response styles:

- **Executive Summary** - concise, decision-oriented insights  
- **Analytical Explanation** - balanced interpretation of dataset patterns *(default)*  
- **Technical Interpretation** - more precise, variable-aware analysis  

This control allows the AI interface to adapt explanations to different user needs while maintaining consistency with the dashboard’s analytical context.